# Ансамбли моделей машинного обучения. Часть 1.

## Подготовка данных

In [1]:
import polars as pl

data = pl.read_csv("Titanic-Dataset.csv").drop("Ticket", "Name")
data

PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
i64,i64,i64,str,f64,i64,i64,f64,str,str
1,0,3,"""male""",22.0,1,0,7.25,null,"""S"""
2,1,1,"""female""",38.0,1,0,71.2833,"""C85""","""C"""
3,1,3,"""female""",26.0,0,0,7.925,null,"""S"""
4,1,1,"""female""",35.0,1,0,53.1,"""C123""","""S"""
5,0,3,"""male""",35.0,0,0,8.05,null,"""S"""
…,…,…,…,…,…,…,…,…,…
887,0,2,"""male""",27.0,0,0,13.0,null,"""S"""
888,1,1,"""female""",19.0,0,0,30.0,"""B42""","""S"""
889,0,3,"""female""",null,1,2,23.45,null,"""S"""


`Cabin` имеет 687 пропусков, лучше данный признак удалить

In [2]:
data = data.drop("Cabin")
data

PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
i64,i64,i64,str,f64,i64,i64,f64,str
1,0,3,"""male""",22.0,1,0,7.25,"""S"""
2,1,1,"""female""",38.0,1,0,71.2833,"""C"""
3,1,3,"""female""",26.0,0,0,7.925,"""S"""
4,1,1,"""female""",35.0,1,0,53.1,"""S"""
5,0,3,"""male""",35.0,0,0,8.05,"""S"""
…,…,…,…,…,…,…,…,…
887,0,2,"""male""",27.0,0,0,13.0,"""S"""
888,1,1,"""female""",19.0,0,0,30.0,"""S"""
889,0,3,"""female""",null,1,2,23.45,"""S"""


In [3]:
from sklearn.preprocessing import LabelEncoder

sex_le = LabelEncoder()

data = data.with_columns(
    Sex=sex_le.fit_transform(data.get_column("Sex")),
)
data

PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
i64,i64,i64,i64,f64,i64,i64,f64,str
1,0,3,1,22.0,1,0,7.25,"""S"""
2,1,1,0,38.0,1,0,71.2833,"""C"""
3,1,3,0,26.0,0,0,7.925,"""S"""
4,1,1,0,35.0,1,0,53.1,"""S"""
5,0,3,1,35.0,0,0,8.05,"""S"""
…,…,…,…,…,…,…,…,…
887,0,2,1,27.0,0,0,13.0,"""S"""
888,1,1,0,19.0,0,0,30.0,"""S"""
889,0,3,0,null,1,2,23.45,"""S"""


In [4]:
data = data.hstack(data.get_column("Embarked").to_dummies()).drop("Embarked")
data

PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S,Embarked_null
i64,i64,i64,i64,f64,i64,i64,f64,u8,u8,u8,u8
1,0,3,1,22.0,1,0,7.25,0,0,1,0
2,1,1,0,38.0,1,0,71.2833,1,0,0,0
3,1,3,0,26.0,0,0,7.925,0,0,1,0
4,1,1,0,35.0,1,0,53.1,0,0,1,0
5,0,3,1,35.0,0,0,8.05,0,0,1,0
…,…,…,…,…,…,…,…,…,…,…,…
887,0,2,1,27.0,0,0,13.0,0,0,1,0
888,1,1,0,19.0,0,0,30.0,0,0,1,0
889,0,3,0,null,1,2,23.45,0,0,1,0


In [5]:
data = data.with_columns(
    Age=pl.col("Age").fill_null(pl.col("Age").median().over("Pclass", "Sex"))
)
data

PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S,Embarked_null
i64,i64,i64,i64,f64,i64,i64,f64,u8,u8,u8,u8
1,0,3,1,22.0,1,0,7.25,0,0,1,0
2,1,1,0,38.0,1,0,71.2833,1,0,0,0
3,1,3,0,26.0,0,0,7.925,0,0,1,0
4,1,1,0,35.0,1,0,53.1,0,0,1,0
5,0,3,1,35.0,0,0,8.05,0,0,1,0
…,…,…,…,…,…,…,…,…,…,…,…
887,0,2,1,27.0,0,0,13.0,0,0,1,0
888,1,1,0,19.0,0,0,30.0,0,0,1,0
889,0,3,0,21.5,1,2,23.45,0,0,1,0


In [6]:
from sklearn.preprocessing import MinMaxScaler

pclass_scaler = MinMaxScaler()
age_scaler = MinMaxScaler()
sibsp_scaler = MinMaxScaler()
parch_scaler = MinMaxScaler()
fare_scaler = MinMaxScaler()

data = data.with_columns(
    Pclass=pl.Series(
        pclass_scaler.fit_transform(data.get_column("Pclass").reshape((-1, 1)))
    ).list.explode(),
    Age=pl.Series(
        age_scaler.fit_transform(data.get_column("Age").reshape((-1, 1)))
    ).list.explode(),
    SibSp=pl.Series(
        sibsp_scaler.fit_transform(data.get_column("SibSp").reshape((-1, 1)))
    ).list.explode(),
    Parch=pl.Series(
        parch_scaler.fit_transform(data.get_column("Parch").reshape((-1, 1)))
    ).list.explode(),
    Fare=pl.Series(
        fare_scaler.fit_transform(data.get_column("Fare").reshape((-1, 1)))
    ).list.explode(),
)
data

PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S,Embarked_null
i64,i64,f64,i64,f64,f64,f64,f64,u8,u8,u8,u8
1,0,1.0,1,0.271174,0.125,0.0,0.014151,0,0,1,0
2,1,0.0,0,0.472229,0.125,0.0,0.139136,1,0,0,0
3,1,1.0,0,0.321438,0.0,0.0,0.015469,0,0,1,0
4,1,0.0,0,0.434531,0.125,0.0,0.103644,0,0,1,0
5,0,1.0,1,0.434531,0.0,0.0,0.015713,0,0,1,0
…,…,…,…,…,…,…,…,…,…,…,…
887,0,0.5,1,0.334004,0.0,0.0,0.025374,0,0,1,0
888,1,0.0,0,0.233476,0.0,0.0,0.058556,0,0,1,0
889,0,1.0,0,0.264891,0.125,0.333333,0.045771,0,0,1,0


In [7]:
data = data.drop("PassengerId")
data

Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S,Embarked_null
i64,f64,i64,f64,f64,f64,f64,u8,u8,u8,u8
0,1.0,1,0.271174,0.125,0.0,0.014151,0,0,1,0
1,0.0,0,0.472229,0.125,0.0,0.139136,1,0,0,0
1,1.0,0,0.321438,0.0,0.0,0.015469,0,0,1,0
1,0.0,0,0.434531,0.125,0.0,0.103644,0,0,1,0
0,1.0,1,0.434531,0.0,0.0,0.015713,0,0,1,0
…,…,…,…,…,…,…,…,…,…,…
0,0.5,1,0.334004,0.0,0.0,0.025374,0,0,1,0
1,0.0,0,0.233476,0.0,0.0,0.058556,0,0,1,0
0,1.0,0,0.264891,0.125,0.333333,0.045771,0,0,1,0


In [8]:
y = data.get_column("Survived")
X = data.drop("Survived")

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

## Обучение ансамблевых моделей

In [10]:
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
)

et = ExtraTreesClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
)

ada = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
)

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=10,
    random_state=42,
)

for ensemble in (rf, et, ada, gb):
    ensemble.fit(X_train, y_train)

    y_pred = ensemble.predict(X_test)
    print(ensemble)
    print(classification_report(y_test, y_pred))

RandomForestClassifier(max_depth=10, random_state=42)
              precision    recall  f1-score   support

           0       0.80      0.89      0.84       110
           1       0.79      0.65      0.71        69

    accuracy                           0.80       179
   macro avg       0.80      0.77      0.78       179
weighted avg       0.80      0.80      0.79       179

ExtraTreesClassifier(max_depth=10, random_state=42)
              precision    recall  f1-score   support

           0       0.77      0.89      0.83       110
           1       0.77      0.58      0.66        69

    accuracy                           0.77       179
   macro avg       0.77      0.74      0.74       179
weighted avg       0.77      0.77      0.76       179

AdaBoostClassifier(learning_rate=0.1, n_estimators=100, random_state=42)
              precision    recall  f1-score   support

           0       0.80      0.85      0.82       110
           1       0.74      0.65      0.69        69

   

|Модель|Accuracy|F1 (для выживших)|
|:-:|:-:|:-:|
|RandomForest (bagging)|**0.80**|0.71|
|ExtraTrees (bagging)|0.77|0.66|
|AdaBoost|0.78|0.69|
|GradientBoosting|**0.80**|**0.74**|

Среди бэггинг‑подходов лучший результат показал случайный лес, тогда как ExtraTrees заметно уступает ему и по accuracy, и по F1 для класса выживших.  

Бустинговые алгоритмы демонстрируют более сбалансированное качество: AdaBoost немного улучшает F1 по сравнению с бэггингом, а градиентный бустинг даёт максимальный F1 для выживших.  

В итоге наилучшим ансамблем для задачи предсказания выживаемости в этом эксперименте является градиентный бустинг, так как он обеспечивает одновременно высокую accuracy и наилучшее качество именно по классу выживших.